# 02 · Link budget — the ray tracer vs. 30 lines of physics

`examples/data/newton_link_metrics.csv` is **real Sionna RT output**: path gain, RMS
delay spread and path count at 14 receiver positions along a radial from the Rising Sun
tower, 3.5 GHz, produced by `ulap-scope analysis`.

The question this notebook asks is the one worth asking before you spend a day of
compute: **how much of that does a simple analytical model already explain — and what,
specifically, does it miss?**

Spoiler: it recovers the path-loss exponent to within 0.05 and tracks the shape to a
fraction of a dB — and it is completely blind to the thing the ray tracer was actually
bought for.

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
EXAMPLES = next(p for p in [here, *here.parents] if (p / "ulap_demo").is_dir())
sys.path[:0] = [str(EXAMPLES), str(EXAMPLES.parent / "ulap-scope")]

import numpy as np
import matplotlib.pyplot as plt
from ulap_demo import (load_scene, load_link_metrics, PropagationModel,
                       fit_log_distance, fspl_db, knife_edge_loss_db)
from ulap_demo.propagation import two_ray_gain_db, wavelength
from ulap_demo.plotting import use_ulap_style, SUNGLOW, MARBLE_WHITE

use_ulap_style()
scene = load_scene()
print(scene.summary())
print()
print(scene.sources())

## What the ray tracer produced

In [ ]:
lm = load_link_metrics()
d, pg = lm["distance_m"], lm["path_gain_dB"]

print(f"{'d [m]':>7} {'path gain [dB]':>15} {'delay spread [ns]':>19} {'paths':>7}")
for i in range(len(d)):
    flag = "  <-- extra ray" if lm["num_paths"][i] > 2 else ""
    print(f"{d[i]:>7.0f} {pg[i]:>15.2f} {lm['delay_spread_ns'][i]:>19.2f} "
          f"{lm['num_paths'][i]:>7.0f}{flag}")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(d, pg, "o-", color=SUNGLOW, lw=1.8, label="Sionna RT path gain")
ax1.set_xlabel("distance from Rising Sun tower [m]")
ax1.set_ylabel("path gain [dB]", color=SUNGLOW)
ax1.grid(alpha=0.25)

ax2 = ax1.twinx()
ax2.plot(d, lm["delay_spread_ns"], "s--", color="#FF7A7A", lw=1.2, alpha=0.9,
         label="RMS delay spread")
ax2.set_ylabel("RMS delay spread [ns]", color="#FF7A7A")
ax1.set_title("Rising Sun link @ 3.5 GHz — ray-traced")
fig.tight_layout()

Two behaviours, very different in character:

- **Path gain** is smooth and almost perfectly log-linear. That is a textbook
  line-of-sight rural link.
- **Delay spread** is near zero (sub-nanosecond) *except* for isolated spikes up to
  361 ns — exactly at the ranges where the path count goes from 2 to 3. A third ray
  appears (a reflection off a building or a terrain facet), and the channel briefly
  becomes dispersive.

Hold that thought: it is where the analytical model will fail.

## Fit the classic log-distance model

$$PL(d) = PL(d_0) + 10\,n\,\log_{10}(d/d_0)$$

$n$ is the **path-loss exponent**: 2 in free space, 3–5 in obstructed/urban settings, and
*below 2* when a ground reflection adds constructively.

In [ ]:
fit = fit_log_distance(d, pg)
print("ray-traced fit:", fit)
print()
print(f"n = {fit.n:.2f}  ->  {'sub-free-space (constructive ground reflection)' if fit.n < 2 else 'obstructed'}")
print(f"RMS residual {fit.rms_db:.2f} dB — the ray tracer's own points barely scatter")

In [ ]:
dd = np.linspace(40, 1450, 300)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(d, pg, "o", color=SUNGLOW, ms=7, label="Sionna RT", zorder=5)
ax.plot(dd, fit.predict(dd), "-", color=SUNGLOW, lw=1.2, alpha=0.6,
        label=f"log-distance fit, n = {fit.n:.2f}")
# free-space reference, anchored to the ray tracer at the first point
anchor = pg[0] + fspl_db(d[0], 3.5e9)
ax.plot(dd, anchor - fspl_db(dd, 3.5e9), "--", color=MARBLE_WHITE, lw=1.1, alpha=0.55,
        label="free space (n = 2), anchored at 50 m")
ax.set_xscale("log")
ax.set_xlabel("distance [m]  (log scale)")
ax.set_ylabel("path gain [dB]")
ax.set_title("Sub-free-space decay: the ground reflection is helping")
ax.grid(alpha=0.25, which="both")
ax.legend()
fig.tight_layout()

The link decays **more slowly than free space**. That is not an error — it is the
two-ray ground reflection. Below the breakpoint

$$d_{bp} = \frac{4 h_t h_r}{\lambda}$$

the direct and ground-reflected rays arrive within less than $\pi$ of each other and add
power on average. For this link:

In [ ]:
h_t, h_r, f = 24.0, 1.5, 3.5e9        # Rising Sun is a 24 m monopole; RX at handset height
d_bp = 4 * h_t * h_r / wavelength(f)
print(f"lambda      = {wavelength(f) * 100:.1f} cm")
print(f"breakpoint  = {d_bp:.0f} m  (the whole transect sits below it)")
print("=> we expect n < 2 across the measured range. Measured n =", round(fit.n, 2))

## Now the analytical model

`ulap_demo.propagation` implements free space + two-ray ground reflection + knife-edge
diffraction over the terrain/building profile.

Before configuring it, notice what the ray tracer already told us: the `num_paths`
column is **2** at ten of the fourteen points. Two paths on a link like this means the
direct ray and its ground reflection — nothing else reached the receiver. So the
geometry along this particular radial was clear, and the physically correct comparison
is the **clear-path two-ray model**, not one carrying terrain and buildings that
demonstrably never entered the answer.

That is not a convenience. It is the data telling us which model applies.

In [ ]:
print(f"points where RT found exactly 2 paths (direct + ground reflection): "
      f"{(lm['num_paths'] == 2).sum()} of {len(d)}")

rising_sun = scene.tower("Rising Sun")
model = PropagationModel(freq_hz=3.5e9, use_terrain=False, use_buildings=False,
                         ground_reflection="average")
print()
print(model.describe())

pg_model = model.radial(scene, rising_sun, d)
fit_model = fit_log_distance(d, pg_model)
print()
print("analytical fit:", fit_model)

### Compare the two

Absolute levels are **not** directly comparable: Sionna's `path_gain` folds in the
antenna patterns (`tr38901` panel at the TX, dipole at the RX) and the array
normalisation, while our model is isotropic-to-isotropic. That is a constant, not a
distance-dependent effect — so the fair comparison is the *shape*, after removing a
single calibration offset.

In [ ]:
offset = float(np.mean(pg - pg_model))
resid = pg - (pg_model + offset)

print(f"path-loss exponent   RT {fit.n:.2f}   vs   analytical {fit_model.n:.2f}"
      f"   (delta {abs(fit.n - fit_model.n):.2f})")
print(f"constant offset      {offset:+.1f} dB  (antenna gain / normalisation)")
print(f"RMS residual after one-constant calibration: {np.sqrt(np.mean(resid ** 2)):.2f} dB")
print(f"worst point:                                 {np.abs(resid).max():.2f} dB")

In [ ]:
fig, (ax, axr) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                              gridspec_kw={"height_ratios": [3, 1]})
ax.plot(d, pg, "o", color=SUNGLOW, ms=7, label="Sionna RT (ground truth)")
ax.plot(d, pg_model + offset, "-", color="#4FC3F7", lw=1.8,
        label=f"analytical model {offset:+.1f} dB")
ax.set_ylabel("path gain [dB]")
ax.set_title("One calibration constant, and the shapes agree")
ax.grid(alpha=0.25); ax.legend()

axr.axhline(0, color=MARBLE_WHITE, lw=0.8, alpha=0.4)
axr.stem(d, resid, linefmt="#4FC3F7", markerfmt="o", basefmt=" ")
axr.set_ylabel("residual [dB]")
axr.set_xlabel("distance from Rising Sun tower [m]")
axr.grid(alpha=0.25)
fig.tight_layout()

Sub-dB agreement on a rural LoS link, from a model that runs in microseconds.

**This is the honest lesson, and it cuts both ways.** For a clear rural radial, the ray
tracer is confirming physics you could have written down. The value of ray tracing is not
here — it is in the places the simple model has no access to.

## Where the simple model is blind

Our model returns *one number per link*: path gain. It has no delay domain at all. The
ray tracer's third ray — the one that produced a 361 ns delay spread at 450 m — simply
does not exist in it.

In [ ]:
spikes = lm["delay_spread_ns"] > 50
print("positions where a third ray appears:")
for dist, ds, npath in zip(d[spikes], lm["delay_spread_ns"][spikes], lm["num_paths"][spikes]):
    coh_bw = 1 / (5 * ds * 1e-9) / 1e6      # rule of thumb: Bc ~ 1/(5 * sigma_tau)
    print(f"  d = {dist:>5.0f} m   delay spread {ds:>6.1f} ns   {npath:.0f} paths"
          f"   -> coherence bandwidth ~{coh_bw:,.0f} MHz")

print("\nAt 100 MHz channel bandwidth, a ~0.5 MHz coherence bandwidth means the channel")
print("is frequency-selective across the carrier: equalisation and scheduling territory.")
print("Our analytical model reports exactly nothing about this.")

That is the buy signal for ray tracing: **delay spread, multipath structure, MIMO rank,
and anything material-dependent.** Use the cheap model to explore, size and sanity-check;
use the ray tracer where dispersion and geometry actually decide the answer.

## Bonus: the fringes the coverage grid can't see

Our coverage maps use `ground_reflection="average"` — the phase-averaged two-ray. Set it
to `"coherent"` and the real interference pattern appears. It is physical, and it is also
why a 10 m coverage grid at 3.5 GHz would alias it into noise.

In [ ]:
dfine = np.linspace(20, 2000, 4000)
coh = two_ray_gain_db(dfine, h_t, h_r, f, mode="coherent")
avg = two_ray_gain_db(dfine, h_t, h_r, f, mode="average")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(dfine, coh, lw=0.6, color="#4FC3F7", alpha=0.75, label="coherent two-ray")
ax.plot(dfine, avg, lw=2.0, color=SUNGLOW, label='breakpoint-smoothed ("average")')
ax.axvline(d_bp, color=MARBLE_WHITE, ls=":", lw=1.2, alpha=0.7)
ax.annotate(f"breakpoint {d_bp:.0f} m", (d_bp, -18), color=MARBLE_WHITE, fontsize=9,
            xytext=(8, 0), textcoords="offset points")
ax.set_xlabel("distance [m]")
ax.set_ylabel("gain relative to free space [dB]")
ax.set_title(f"Two-ray ground reflection, {h_t:.0f} m TX -> {h_r} m RX @ 3.5 GHz")
ax.set_ylim(-25, 10); ax.grid(alpha=0.25); ax.legend(loc="lower left")
fig.tight_layout()

## Bonus: a link budget you can defend

Given an EIRP, a noise floor and a required SNR, how far does a cell reach — and what
does moving band cost you? This is the λ² penalty made concrete.

In [ ]:
from ulap_demo import noise_floor_dbm

REQUIRED_SNR_DB = 3.0
BW = 100e6
noise = noise_floor_dbm(BW, noise_figure_db=7.0)
sensitivity = noise + REQUIRED_SNR_DB
print(f"noise floor: {noise:.1f} dBm over {BW/1e6:.0f} MHz (NF 7 dB)")
print(f"required RX power: {sensitivity:.1f} dBm\n")

dgrid = np.logspace(1, 4.5, 2000)

def reach_m(ghz, ground_reflection):
    m = PropagationModel(freq_hz=ghz * 1e9, use_terrain=False)
    gain = (two_ray_gain_db(dgrid, h_t, h_r, ghz * 1e9, mode=ground_reflection)
            - fspl_db(dgrid, ghz * 1e9))
    ok = dgrid[m.eirp_dbm + gain >= sensitivity]
    return float(ok.max()) if ok.size else float("nan")

print(f"{'band':>8} {'d_bp':>8} {'reach: free space':>19} {'reach: + ground ray':>21}")
for ghz in (1.8, 3.5, 6.0, 10.0, 28.0):
    d_bp_f = 4 * h_t * h_r / wavelength(ghz * 1e9)
    print(f"{ghz:>6} GHz {d_bp_f:>6.0f} m {reach_m(ghz, 'none'):>16.0f} m "
          f"{reach_m(ghz, 'average'):>18.0f} m")

Read those two columns against each other, because they tell different stories.

**Free space alone** behaves exactly as the textbook promises: reach $\propto 1/f$, so
site count $\propto f^2$. Going from 1.8 to 10 GHz costs you ~5.5× the sites for the same
area.

**Add the ground ray and the ordering partly collapses.** Beyond the breakpoint the two
rays cancel progressively and loss steepens toward $d^{-4}$ — but the breakpoint
$d_{bp} = 4h_th_r/\lambda$ *itself* scales with frequency, so a high band stays in its
gentle $n\approx2$ region longer. The low band hits its $d^{-4}$ wall first.

This is the point of building a twin instead of quoting a rule of thumb: the rule of
thumb ("high band = short range") is directionally right and quantitatively unreliable,
and which effect dominates depends on your antenna heights, not on the band alone. The
pilot's ray-traced sweep (`docs/renders/sweep_frequency.png`) tracks free-space λ² physics
to within ~1 dB over a real scene, and notebook 04 shows where that stops being true —
at the **cell edge**, which is the part of the distribution that decides your site count.

> ⚠️ These are clear-path ranges over a **flat, curvature-free earth**. No terrain
> shadowing, no buildings, no rain, no foliage, and no oxygen absorption (which alone
> would cost 60 GHz ~15 dB/km). The double-digit-kilometre entries are also past the point
> where the radio horizon (~25 km for these antenna heights) starts to be the real limit —
> they say "not power-limited", not "will work".

## Next

- **[03 · The scene and its terrain](03_scene_and_terrain.ipynb)** — what the twin is
  actually made of, and why receiver height is a planning decision.
- **[04 · Coverage & SINR](04_coverage_and_sinr.ipynb)** — from one radial to whole-area maps.